<a id="top"></a>

# Lab L0709: validate tool calls (the application validates)

```yaml
title: "Lab L0709: validate tool calls (the application validates)"
keywords: validation, hallucinated tool call, dispatch, missing argument, invented tool, protocol, lab
estimated duration: 25
```

> **Lesson:** L07. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L07/objectives.md). Solutions: `L0409_lab_solutions.ipynb`. **Pure Python — no API key needed** (you validate crafted `tool_use` blocks, the same offline approach as the L0708 demo's fallback cell).
>
> **After this lab you can:** write the validation step that catches a hallucinated tool call, and explain why the schema alone does not prevent one.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Write the validator](#2-problem-1--write-the-validator)
- [3. Problem 2 — Run it over every crafted call](#3-problem-2--run-it-over-every-crafted-call)
- [4. Problem 3 — The three outcomes (written)](#4-problem-3--the-three-outcomes-written)
- [5. Problem 4 — Why doesn't the schema stop a bad call? (written)](#5-problem-4--why-doesnt-the-schema-stop-a-bad-call-written)
- [6. Self-check](#6-self-check)

## 1. Setup

The `calculator` (which raises on non-arithmetic input), plus crafted `tool_use`-like objects — one good, three hallucinated — mimicking what a model might emit.

In [1]:
from types import SimpleNamespace


def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


CALCULATOR_TOOL: dict[str, object] = {
    "name": "calculator",
    "description": (
        "Evaluate a basic arithmetic expression (the four operators and parentheses) and "
        "return the exact numeric result. Use this whenever the user asks for a calculation."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '18374 * 92431'."}},
        "required": ["expression"],
    },
}


# Crafted tool calls a model might emit. SimpleNamespace mimics the SDK's block
# (attribute access: .name, .input).
CALLS = [
    SimpleNamespace(name="calculator", input={"expression": "47219 * 8803"}),  # good
    SimpleNamespace(name="calculator", input={"expression": "twenty + 1"}),  # non-numeric
    SimpleNamespace(name="calculator", input={}),  # missing arg
    SimpleNamespace(name="wikipedia", input={"query": "geese"}),  # invented tool
]
print(len(CALLS), "crafted calls ready")

4 crafted calls ready


[↑ Back to top](#top)

## 2. Problem 1 — Write the validator

Implement `validate_call(call)` returning an `"OK: ..."` string with the result for a good call, or a `"REJECTED: ..."` string for each failure: an unknown tool name, a missing `expression` argument, or a `calculator` `ValueError`. The application — not the model — is responsible for catching all three.

In [2]:
def validate_call(call: object) -> str:
    """Validate a tool_use block and run it, or return a loud rejection string."""
    if call.name != "calculator":
        return f"REJECTED: no such tool {call.name!r} (the model invented it)"
    if "expression" not in call.input:
        return f"REJECTED: missing required argument 'expression' (got {call.input!r})"
    try:
        return (
            f"OK: calculator({call.input['expression']!r}) = {calculator(call.input['expression'])}"
        )
    except ValueError as exc:
        return f"REJECTED: {exc}"


print(validate_call(CALLS[0]))  # expect OK with a number

OK: calculator('47219 * 8803') = 415668857


[↑ Back to top](#top)

## 3. Problem 2 — Run it over every crafted call

Loop over `CALLS` and print the verdict for each. Three of the four should be REJECTED.

In [3]:
for call in CALLS:
    print(validate_call(call))

OK: calculator('47219 * 8803') = 415668857
REJECTED: refusing to evaluate non-arithmetic expression: 'twenty + 1'
REJECTED: missing required argument 'expression' (got {})
REJECTED: no such tool 'wikipedia' (the model invented it)


[↑ Back to top](#top)

## 4. Problem 3 — The three outcomes (written)

Name the **three** observable outcomes of handing a model a single tool, in one line each.

_Write your answer by editing this cell (double-click)._

1. _TODO_
2. _TODO_
3. _TODO_

[↑ Back to top](#top)

## 5. Problem 4 — Why doesn't the schema stop a bad call? (written)

"The application validates; the model proposes." In a sentence or two: why does passing a JSON-Schema tool definition *not* prevent the model from emitting malformed or invented arguments?

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0409_lab_solutions.ipynb`. Done when `validate_call` returns `OK` for the first call and `REJECTED` (with a useful reason) for the non-numeric, missing-arg, and invented-tool calls.

[↑ Back to top](#top)